# EmpowerLens — Flat Mental-RoBERTa Experiment Suite (Kaggle GPU runner)

Runs Experiments 1–8 from `src/experiments_flat_mentalroberta.py` on Kaggle's free T4 GPU.
**Cascade is NOT run here** — this notebook is flat-architecture only (see the script's module docstring).

**Before running:**
1. Settings → **Accelerator: GPU (T4 x2 is fine, we pin to GPU 0 below)**, **Internet: On**.
2. `src/experiments_flat_mentalroberta.py` must already be **committed and pushed** to the `lumia-space` branch
   (copy it from this project folder into `EmpowerLens/src/` in your repo, commit, push — then this notebook just clones it).
3. Frozen splits (`data/splits`, `data/splits_codipas_cls`, `data/splits_combined`) must already be committed and pushed too.
4. Run cells top to bottom. Each experiment cell is independent — skip ones you don't need yet, but run
   Experiment 1 first since it decides whether Experiments 3/4/5 are even necessary.
5. Kaggle sessions are killed at 12h — Experiments 2 and 7 (3 configs/seeds × training) are the heaviest;
   consider running them in separate sessions if you're tight on time.


## 0. Clone the repo, install deps, pin to a single GPU
Pinning `CUDA_VISIBLE_DEVICES=0` avoids the T4×2 cross-device deadlock documented in project notes.

In [ ]:
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Sanity check: the experiment script must be present at src/experiments_flat_mentalroberta.py
import os
assert os.path.exists("src/experiments_flat_mentalroberta.py"), (
    "src/experiments_flat_mentalroberta.py not found — commit it to the lumia-space branch "
    "and push before running this notebook."
)
print("Found src/experiments_flat_mentalroberta.py")


## 1. Experiment 1 — Data / label audit
No training needed unless you already have checkpoints to audit. Fast, run this first.

In [ ]:
!python -m src.experiments_flat_mentalroberta --experiment 1 \
    --splits data/splits_combined \
    --out results/exp1

# Optional, if you already have trained checkpoints to audit per-class + confusion matrix:
# !python -m src.experiments_flat_mentalroberta --experiment 1 \
#     --splits data/splits_combined --out results/exp1 \
#     --checkpoint-mc checkpoints/multiclass_mental-roberta-base_42 \
#     --checkpoint-ml checkpoints/multilabel_mental-roberta-base_42


## 2. Experiment 2 — Dataset ablation (Annotated vs CODIPAS vs combined)

**Runs ONE dataset at a time, not in parallel.** Each cell below trains exactly one
data config (all 3 seeds for that config, sequentially) and evaluates it, then frees the GPU
before the next cell starts. Run them in order, or spread them across separate Kaggle sessions —
results accumulate in the same CSV either way. Task is **multilabel** (the production task).

All outputs for this experiment go to `result_experiment/exp2/` (created automatically).


### 2a. Annotated only

In [ ]:
!python -m src.experiments_flat_mentalroberta --experiment 2 \
    --task multilabel \
    --out result_experiment/exp2 \
    --only-config annotated_only


### 2b. CODIPAS only

In [ ]:
!python -m src.experiments_flat_mentalroberta --experiment 2 \
    --task multilabel \
    --out result_experiment/exp2 \
    --only-config codipas_only


### 2c. Annotated + CODIPAS (combined)

In [ ]:
!python -m src.experiments_flat_mentalroberta --experiment 2 \
    --task multilabel \
    --out result_experiment/exp2 \
    --only-config annotated_plus_codipas


### 2d. Aggregate (run after all three of the above have finished — no retraining, just re-reads the CSVs)

In [ ]:
!python -m src.experiments_flat_mentalroberta --experiment 2 \
    --task multilabel \
    --out result_experiment/exp2 \
    --aggregate-only

import pandas as pd
print(pd.read_csv('result_experiment/exp2/exp2_all_seed_results.csv'))


## 3. Experiment 3 — Multiclass imbalance: standard CE vs class-weighted CE
Only worth running if Experiment 1's audit showed meaningful class imbalance.

In [ ]:
!python -m src.experiments_flat_mentalroberta --experiment 3 \
    --task multiclass \
    --splits data/splits_combined \
    --out results/exp3


## 4. Experiment 4 — Focal loss vs class-balanced loss
Only run if Experiment 3's weighted-CE gain over plain CE looks insufficient.

In [ ]:
!python -m src.experiments_flat_mentalroberta --experiment 4 \
    --task multiclass \
    --splits data/splits_combined \
    --out results/exp4 \
    --gamma 2.0 --cb-beta 0.999


## 5. Experiment 5 — Weighted sampling vs best loss-based approach
Set `--best-loss` to whichever loss won in Experiment 3/4 (e.g. `weighted_ce`, `focal`, or `class_balanced`).

In [ ]:
BEST_LOSS = "weighted_ce"  # <-- update after reading results/exp3 and results/exp4

!python -m src.experiments_flat_mentalroberta --experiment 5 \
    --task multiclass \
    --splits data/splits_combined \
    --out results/exp5 \
    --best-loss $BEST_LOSS


## 6. Experiment 6 — Multilabel per-label performance + label-wise loss weighting
Thresholds are swept on val only (inside train_transformer.py's sweep_thresholds) and then fixed for test.

In [ ]:
!python -m src.experiments_flat_mentalroberta --experiment 6 \
    --task multilabel \
    --splits data/splits_combined \
    --out results/exp6 \
    --loss weighted_bce

# To try label-wise focal weighting instead:
# !python -m src.experiments_flat_mentalroberta --experiment 6 \
#     --task multilabel --splits data/splits_combined --out results/exp6 \
#     --loss focal --gamma 2.0


## 7. Experiment 7 — Flat multilabel report
**Flat architecture only.** No cascade run or comparison happens here — Izza's cascade track is separate.

In [ ]:
!python -m src.experiments_flat_mentalroberta --experiment 7 \
    --task multilabel \
    --splits data/splits_combined \
    --out results/exp7 \
    --loss weighted_bce


## 8. Experiment 8 — Sequence length / truncation analysis
Only launches a Longformer comparison run if truncation at 512 tokens exceeds the threshold AND `--run-longformer` is passed.

In [ ]:
!python -m src.experiments_flat_mentalroberta --experiment 8 \
    --splits data/splits_combined \
    --out results/exp8 \
    --truncation-threshold-pct 10

# If exp8 reports substantial truncation at 512, re-run with:
# !python -m src.experiments_flat_mentalroberta --experiment 8 \
#     --splits data/splits_combined --out results/exp8 --task multiclass \
#     --run-longformer --longformer-model allenai/longformer-base-4096 \
#     --seed 42 --loss weighted_ce


## 9. Copy everything to /kaggle/working so it's downloadable

In [ ]:
!mkdir -p /kaggle/working/results /kaggle/working/result_experiment
!cp -r results/* /kaggle/working/results/ 2>/dev/null || true
!cp -r result_experiment/* /kaggle/working/result_experiment/ 2>/dev/null || true
!ls -la /kaggle/working/results
!ls -la /kaggle/working/result_experiment
